# Kaggle Notebook - Question 3 
 
This notebook runs Q3 using the strict two-pass flow with manual review.

In [ ]:
import os 
import subprocess 
import sys 
 
REPO_DIR = '/kaggle/working/Hindi-ASR-Whisper-Pipeline' 
REPO_URL = 'https://github.com/GyanendraChaubey/Hindi-ASR-Whisper-Pipeline.git' 
 
if not os.path.exists(REPO_DIR): 
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True) 
 
os.chdir(REPO_DIR) 
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True) 
print('Setup complete. Current dir:', os.getcwd())

## Pass 1: generate low-confidence sample 
 
In strict mode, this command can exit non-zero after creating the sample CSV.

In [ ]:
manifest_csv = 'data/FT Data - data.csv' 
wordlist_file = 'data/unique_words.csv'  # If unavailable, set to None 
 
cmd = [ 
    sys.executable, 'scripts/run_question3.py', 
    '--manifest-csv', manifest_csv, 
    '--work-dir', 'artifacts/q3', 
    '--review-sample-size', '50', 
] 
if wordlist_file: 
    cmd.extend(['--wordlist-file', wordlist_file]) 
 
print('Running pass-1:', ' '.join(cmd)) 
result = subprocess.run(cmd, check=False) 
print('Return code:', result.returncode)

In [ ]:
import pandas as pd 
from pathlib import Path 
 
sample_path = Path('artifacts/q3/low_confidence_review_sample.csv') 
print('Sample file exists:', sample_path.exists()) 
if sample_path.exists(): 
    df = pd.read_csv(sample_path) 
    print('Rows:', len(df)) 
    display(df.head(10))

## Manual step 
 
Fill manual_label as correct or incorrect for 40-50 rows in the sample CSV. 
Then run pass-2 below.

In [ ]:
manual_review_file = 'artifacts/q3/low_confidence_review_sample.csv' 
# Example Kaggle input path: 
# manual_review_file = '/kaggle/input/<your-dataset>/low_confidence_review_sample.csv' 
 
cmd = [ 
    sys.executable, 'scripts/run_question3.py', 
    '--manifest-csv', 'data/FT Data - data.csv', 
    '--work-dir', 'artifacts/q3', 
    '--review-sample-size', '50', 
    '--manual-review-file', manual_review_file, 
] 
if wordlist_file: 
    cmd.extend(['--wordlist-file', wordlist_file]) 
 
print('Running pass-2:', ' '.join(cmd)) 
subprocess.run(cmd, check=True)

In [ ]:
from pathlib import Path 
 
work_dir = Path('artifacts/q3') 
expected = [ 
    work_dir / 'word_classification_with_confidence.csv', 
    work_dir / 'google_sheet_ready_word_labels.csv', 
    work_dir / 'low_confidence_review_sample.csv', 
    work_dir / 'low_confidence_review_analysis.json', 
    work_dir / 'summary.json', 
    work_dir / 'question3_report.md', 
] 
 
for p in expected: 
    print('OK' if p.exists() else 'MISSING', p)